In [1]:
import copy
import math

X = "X"
O = "O"
EMPTY = None

# Cấu hình kích thước N
N = 5  # Bạn có thể đổi thành 10 hoặc bất kỳ số nào
WIN_LENGTH = 4  # Số quân liên tiếp để thắng (thường là 4 hoặc 5 với bàn cờ lớn)
MAX_DEPTH = 3   # Giới hạn độ sâu để Minimax không bị treo máy

def initial_state(n):
    return [[EMPTY for _ in range(n)] for _ in range(n)]

def player(turn_count, user_char):
    ai_char = O if user_char == X else X
    return user_char if turn_count % 2 == 0 else ai_char

def actions(board):
    return {(i, j) for i in range(N) for j in range(N) if board[i][j] == EMPTY}

def result(board, action, p_char):
    i, j = action
    new_board = copy.deepcopy(board)
    new_board[i][j] = p_char
    return new_board

def winner(board):
    # Kiểm tra hàng dọc, ngang, chéo cho N x N
    for i in range(N):
        for j in range(N):
            if board[i][j] == EMPTY: continue
            curr = board[i][j]
            # Kiểm tra ngang
            if j <= N - WIN_LENGTH and all(board[i][j+k] == curr for k in range(WIN_LENGTH)): return curr
            # Kiểm tra dọc
            if i <= N - WIN_LENGTH and all(board[i+k][j] == curr for k in range(WIN_LENGTH)): return curr
            # Kiểm tra chéo xuôi
            if i <= N - WIN_LENGTH and j <= N - WIN_LENGTH and all(board[i+k][j+k] == curr for k in range(WIN_LENGTH)): return curr
            # Kiểm tra chéo ngược
            if i <= N - WIN_LENGTH and j >= WIN_LENGTH - 1 and all(board[i+k][j-k] == curr for k in range(WIN_LENGTH)): return curr
    return None

def terminal(board):
    return winner(board) is not None or not any(EMPTY in row for row in board)

def evaluate(board, user_char):
    """Hàm đánh giá trạng thái khi chưa kết thúc game (Heuristic)"""
    res = winner(board)
    ai_char = O if user_char == X else X
    if res == ai_char: return 1000
    if res == user_char: return -1000
    return 0 # Bạn có thể viết thêm logic đếm chuỗi 2, 3 quân ở đây

# --- Thuật toán Minimax giới hạn độ sâu ---
def minimax(board, user_char):
    if terminal(board): return None
    ai_char = O if user_char == X else X

    best_val = -math.inf
    move = None

    # Ưu tiên các ô ở giữa bàn cờ để tối ưu tốc độ và hiệu quả
    possible_actions = sorted(list(actions(board)), key=lambda x: abs(x[0]-N/2) + abs(x[1]-N/2))

    for a in possible_actions:
        val = minValue(result(board, a, ai_char), user_char, 0)
        if val > best_val:
            best_val = val
            move = a
    return move

def maxValue(state, user_char, depth):
    if terminal(state) or depth >= MAX_DEPTH:
        return evaluate(state, user_char)

    v = -math.inf
    ai_char = O if user_char == X else X
    for action in actions(state):
        v = max(v, minValue(result(state, action, ai_char), user_char, depth + 1))
    return v

def minValue(state, user_char, depth):
    if terminal(state) or depth >= MAX_DEPTH:
        return evaluate(state, user_char)

    v = math.inf
    for action in actions(state):
        v = min(v, maxValue(result(state, action, user_char), user_char, depth + 1))
    return v

def print_board(board):
    print("\n   " + " ".join(f"{i:2}" for i in range(N)))
    print("   " + "---" * N)
    for i, row in enumerate(board):
        row_str = f"{i:1} |"
        for cell in row:
            char = cell if cell else "."
            row_str += f" {char}  "
        print(row_str)

if __name__ == "__main__":
    N = int(input("Nhập kích thước bàn cờ N: "))
    WIN_LENGTH = 5 if N >= 5 else 3 # Thường bàn cờ lớn cần 5 quân
    MAX_DEPTH = 2 if N > 5 else 3   # Giảm độ sâu nếu bàn cờ quá lớn

    board = initial_state(N)
    user_char = input("Bạn chọn X hay O? ").upper()
    turn = 0

    while not terminal(board):
        print_board(board)
        curr_p = player(turn, user_char)

        if curr_p == user_char:
            try:
                r = int(input(f"Nhập hàng (0-{N-1}): "))
                c = int(input(f"Nhập cột (0-{N-1}): "))
                if board[r][c] != EMPTY: raise ValueError
                board[r][c] = user_char
            except:
                print("Ô không hợp lệ!")
                continue
        else:
            print("AI đang tính toán...")
            move = minimax(board, user_char)
            board[move[0]][move[1]] = curr_p
        turn += 1

    print_board(board)
    res = winner(board)
    print(f"KẾT THÚC: {res} THẮNG!" if res else "HÒA!")

Nhập kích thước bàn cờ N: 5
Bạn chọn X hay O? o

    0  1  2  3  4
   ---------------
0 | .   .   .   .   .  
1 | .   .   .   .   .  
2 | .   .   .   .   .  
3 | .   .   .   .   .  
4 | .   .   .   .   .  
Nhập hàng (0-4): 1
Nhập cột (0-4): 1

    0  1  2  3  4
   ---------------
0 | .   .   .   .   .  
1 | .   O   .   .   .  
2 | .   .   .   .   .  
3 | .   .   .   .   .  
4 | .   .   .   .   .  
AI đang tính toán...

    0  1  2  3  4
   ---------------
0 | .   .   .   .   .  
1 | .   O   .   .   .  
2 | .   .   X   .   .  
3 | .   .   .   .   .  
4 | .   .   .   .   .  
Nhập hàng (0-4): 5
Nhập cột (0-4): 3
Ô không hợp lệ!

    0  1  2  3  4
   ---------------
0 | .   .   .   .   .  
1 | .   O   .   .   .  
2 | .   .   X   .   .  
3 | .   .   .   .   .  
4 | .   .   .   .   .  
Nhập hàng (0-4): 0
Nhập cột (0-4): 0

    0  1  2  3  4
   ---------------
0 | O   .   .   .   .  
1 | .   O   .   .   .  
2 | .   .   X   .   .  
3 | .   .   .   .   .  
4 | .   .   .   .   .  
AI đang tính t